In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.common.exceptions import NoSuchElementException, TimeoutException
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time

In [2]:
options = Options()
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--window-size=1920,1080")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")

Printers = []
seen_links = set()  

In [3]:
START_URL = "https://www.amazon.eg/s?k=printer&i=electronics&s=exact-aware-popularity-rank&crid=1S1DHOM46XWW3&qid=1794142248&sprefix=pri%2Celectronics%2C218&ref=sr_st_exact-aware-popularity-rank&ds=v1%3AzurmoWyENWcWGrXXYR4kOop2MLZaksOtWI03DiIdXCA"

driver = webdriver.Chrome(options=options)
driver.get(START_URL)

## -- Accept Cookies --
try:
    WebDriverWait(driver, 5).until(
        EC.element_to_be_clickable((By.ID, "sp-cc-accept"))
    ).click()
except TimeoutException:
    pass

## -- Click "See all results" only if it exists (category pages only) --
try:
    WebDriverWait(driver, 3).until(
        EC.element_to_be_clickable((By.ID, "apb-desktop-browse-search-see-all"))
    ).click()
except TimeoutException:
    pass

Error sending stats to Plausible: error sending request for url (https://plausible.io/api/event)


In [4]:
## -- Scraping --
page = 1

while True:
    try:
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
            )
        )
    except TimeoutException:
        print("No results found on this page (maybe a CAPTCHA?). Stopping.")
        break

    cards = driver.find_elements(By.CSS_SELECTOR, 'div[data-component-type="s-search-result"]')
    page_products = 0

    for card in cards:
        try:
            Name = card.find_element(By.CSS_SELECTOR, 'h2').get_attribute('aria-label')
            Link = card.find_element(By.CSS_SELECTOR, 'a.a-link-normal[href*="/dp/"]').get_attribute('href')
            Image = card.find_element(By.CSS_SELECTOR, 'img.s-image').get_attribute('src')
        except NoSuchElementException:
            continue

        clean_link = Link.split("?")[0]
        if clean_link in seen_links:
            continue
        seen_links.add(clean_link)

        try:
            Rating = card.find_element(By.CSS_SELECTOR, '[data-cy="reviews-block"] span.a-size-small.a-color-base').text
        except NoSuchElementException:
            Rating = "Not Available"

        try:
            Reviews = card.find_element(By.CSS_SELECTOR, '[data-cy="reviews-block"] a[aria-label$="ratings"]').get_attribute('aria-label')
        except NoSuchElementException:
            Reviews = "Not Available"

        try:
            Price = card.find_element(By.CLASS_NAME, 'a-price-whole').text
        except NoSuchElementException:
            Price = "Product Not Available"

        Printers.append({
            'Name': Name,
            'Link': clean_link,
            'Image': Image,
            'Rating': Rating,
            'Reviews': Reviews,
            'Price': Price
        })
        page_products += 1

    print(
        f"Page {page} finished successfully. "
        f"Products collected on this page: {page_products}. "
        f"Total products collected: {len(Printers)}"
    )

    try:
        next_page = driver.find_element(By.CSS_SELECTOR, 'a.s-pagination-next')

        if 's-pagination-disabled' in next_page.get_attribute('class'):
            print("Scraping finished. No more pages available.")
            break

        next_page.click()
        page += 1
        time.sleep(5)

    except NoSuchElementException:
        print("Next page button not found. Check the pagination selector.")
        break

driver.quit()

Page 1 finished successfully. Products collected on this page: 24. Total products collected: 24
Page 2 finished successfully. Products collected on this page: 20. Total products collected: 44
Page 3 finished successfully. Products collected on this page: 24. Total products collected: 68
Page 4 finished successfully. Products collected on this page: 0. Total products collected: 68
Page 5 finished successfully. Products collected on this page: 24. Total products collected: 92
Page 6 finished successfully. Products collected on this page: 24. Total products collected: 116
Page 7 finished successfully. Products collected on this page: 24. Total products collected: 140
Page 8 finished successfully. Products collected on this page: 24. Total products collected: 164
Page 9 finished successfully. Products collected on this page: 24. Total products collected: 188
Page 10 finished successfully. Products collected on this page: 24. Total products collected: 212
Page 11 finished successfully. Prod

In [5]:
df_Printers = pd.DataFrame(Printers)
df_Printers.head()

,Name,Link,Image,Rating,Reviews,Price
0,"EPSON EcoTank L3250 Home ink tank printer A4, ...",https://www.amazon.eg/-/en/EcoTank-SmartPanel-...,https://m.media-amazon.com/images/I/818P+qYvSH...,4.3,85 ratings,"8,499"
1,"HP Smart Tank 585 Wireless All In One Printer,...",https://www.amazon.eg/-/en/HP-Smart-Wireless-P...,https://m.media-amazon.com/images/I/61KRklpNnG...,3.6,169 ratings,"7,199"
2,Brother DCP-T530DW | Affordable all-in-one pri...,https://www.amazon.eg/-/en/Brother-DCP-T530DW-...,https://m.media-amazon.com/images/I/61Ix+P4uH0...,4.0,48 ratings,"8,499"
3,Canon IJ MFP PIXMA G3430,https://www.amazon.eg/-/en/Canon-IJ-MFP-PIXMA-...,https://m.media-amazon.com/images/I/51KoI+bkxY...,3.8,320 ratings,"6,899"
4,"Canon Pixma TS3640 Printer - Black, 1 Years Of...",https://www.amazon.eg/-/en/Canon-Pixma-TS3640-...,https://m.media-amazon.com/images/I/61liGslfAK...,3.7,371 ratings,Product Not Available


In [6]:
df_Printers.to_csv("amazon_eg_printers.csv", index=False, encoding="utf-8-sig")

In [1]:
import pandas as pd
df = pd.read_csv("amazon_eg_printers.csv", encoding="utf-8-sig")
df.to_parquet("amazon_eg_printers.parquet", index=False)